# Job Market EDA Pipeline
Connects to MongoDB Atlas to retrieve scraped job data, removes duplicates, and generates exploratory data visualizations.

In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Configuration
DB_NAME = "job_market_db"
COLLECTION_NAME = "jobs_raw"

# Ensure pandas outputs standard dataframes
pd.set_option('display.max_columns', None)

## 1. Load Data from MongoDB

In [3]:
mongo_uri = os.getenv("MONGO_URI")
if not mongo_uri:
    raise ValueError("MONGO_URI is missing. Check your .env file.")

print("🔗 Connecting to MongoDB...")
client = MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

print(f"📥 Fetching records from '{COLLECTION_NAME}'...")
# Exclude the MongoDB internal _id to keep the dataframe clean
cursor = collection.find({}, {"_id": 0})

df = pd.DataFrame(list(cursor))
print(f"✅ Loaded {len(df)} jobs into Pandas DataFrame.")

# Display the first few rows
df.head()

🔗 Connecting to MongoDB...
📥 Fetching records from 'jobs_raw'...
✅ Loaded 3000 jobs into Pandas DataFrame.


,origin_id,source_api,job_title,company_name,location,is_remote,seniority_level,description,tech_stack
0,5929145288,Adzuna,Senior Data Engineer (m/w/d),Real I.S. AG Gesellschaft für Immobilien Asset...,"Bayern, Au-Haidhausen",False,Senior,Stellenbeschreibung: Die Real I.S. Gruppe ist ...,[]
1,5900622139,Adzuna,Data Engineer / Data Platform Architect (w/m/d),Bausparkasse Mainz AG,"Rheinland-Pfalz, Mainz",False,Unspecified,Die Bausparkasse Mainz AG (BKM) ist ein Finanz...,[]
2,5929145113,Adzuna,Cloud Data Engineer (m/w/d) – AWS und Databricks,VGH Versicherungen,"Niedersachsen, Bult",False,Unspecified,Einleitung Wir suchen Sie als engagierten und ...,[aws]
3,5857239128,Adzuna,"Data Center Chief Engineer, DCEO",Amazon Workforce Staffing,"Hessen, Frankfurt am Main",False,Unspecified,"Amazon is looking for an energetic, detail-ori...",[aws]
4,5858241512,Adzuna,Data Engineer,Nespresso Deutschland GmbH,Deutschland,True,Unspecified,POSITION SNAPSHOT Location: Bangkok Company: N...,[]


In [4]:
df.tail()

,origin_id,source_api,job_title,company_name,location,is_remote,seniority_level,description,tech_stack
2995,12265-294814_JB5269923-S,Jobsuche,Data Analyst (m/w/d),FERCHAU GmbH Niederlassung Rostock,Rostock,False,Unspecified,"Technologien auf das nächste Level zu heben, s...",[]
2996,10001-1003685161-S,Jobsuche,Data Science Engineer m/w/d,Krannich Solar GmbH & Co. KG,Weil der Stadt,False,Unspecified,### DAS IST IHR JOB\n\nUnsere Abteilung Softwa...,[]
2997,13319-895867/1_624389LS-S,Jobsuche,Senior Data Engineer (m/w/d),Hays Professional Solutions GmbH Standort Dort...,Salzgitter,False,Senior,\n\n### Ihre Vorteile:\n\n- Betreuung im gesam...,[]
2998,12362-PR-292039_624432LS-S,Jobsuche,Data Software Engineer (w/m/d),Brunel GmbH NL Rosenheim,"Rosenheim, Oberbayern",False,Unspecified,### Ihre Aufgaben\n\nMöchten Sie die nächste S...,[python]
2999,12634-BB-646266-7878-1255-S,Jobsuche,Data Engineer (m/w/d) mit Schwerpunkt Test und...,Akkodis Germany Tech Experts GmbH,Frankfurt am Main,False,Unspecified,Für ein langfristig angelegtes Projekt im Bere...,"[python, sql, aws]"


## 2. Data Cleaning and Deduplication

In [5]:
initial_count = len(df)

# Deduplicate based on core fields to avoid dropping slightly altered re-posts
subset_keys = [col for col in ['title', 'company', 'location'] if col in df.columns]

if subset_keys:
    df_cleaned = df.drop_duplicates(subset=subset_keys, keep='first').copy()
else:
    df_cleaned = df.drop_duplicates().copy()
    
dropped_count = initial_count - len(df_cleaned)
print(f"🧹 Removed {dropped_count} duplicate records.")
print(f"📊 Remaining records: {len(df_cleaned)}")

# Fill missing string values
df_cleaned = df_cleaned.fillna("Unknown")

🧹 Removed 2527 duplicate records.
📊 Remaining records: 473


## 3. Exploratory Data Analysis (EDA)

In [6]:
# Display dataset statistics and data types
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 473 entries, 0 to 2995
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   origin_id        473 non-null    object
 1   source_api       473 non-null    object
 2   job_title        473 non-null    object
 3   company_name     473 non-null    object
 4   location         473 non-null    object
 5   is_remote        473 non-null    bool  
 6   seniority_level  473 non-null    object
 7   description      473 non-null    object
 8   tech_stack       473 non-null    object
dtypes: bool(1), object(8)
memory usage: 33.7+ KB


In [9]:
df_cleaned.describe()

,origin_id,source_api,job_title,company_name,location,is_remote,seniority_level,description,tech_stack
count,473,473,473,473,473,473,473,473,473
unique,473,2,317,379,473,2,5,440,96
top,5929145288,Adzuna,Data Engineer (m/w/d),Instaffo GmbH,"Bayern, Au-Haidhausen",False,Unspecified,"At Fresenius Medical Care, we are the global l...",[]
freq,1,239,38,10,1,435,371,5,300


In [7]:
if 'company' in df_cleaned.columns:
    top_companies = df_cleaned['company'].value_counts().head(10)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(y=top_companies.index, x=top_companies.values, palette="viridis")
    plt.title("Top 10 Companies by Job Postings")
    plt.xlabel("Number of Postings")
    plt.ylabel("Company")
    plt.tight_layout()
    plt.show()

In [8]:
if 'title' in df_cleaned.columns:
    top_titles = df_cleaned['title'].value_counts().head(10)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(y=top_titles.index, x=top_titles.values, palette="mako")
    plt.title("Top 10 Most Frequent Job Titles")
    plt.xlabel("Number of Postings")
    plt.ylabel("Job Title")
    plt.tight_layout()
    plt.show()